# Audio File Sample Rate & Duration
Provide a folder path and this notebook will list audio files with their sample rate and duration.

## 1) Import Libraries and Configure Paths

In [1]:
from pathlib import Path
import pandas as pd
import soundfile as sf

# Set this to the folder containing your audio files
input_path = Path("../data/classical_resample")
# Optional: output CSV path (set to None to skip saving)
output_csv = Path("../data/audio_metadata3.csv")

## 2) Validate Input Path and Discover Audio Files

In [2]:
audio_exts = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aiff", ".aif"}

if not input_path.exists():
    raise FileNotFoundError(f"Input path does not exist: {input_path}")

audio_files = [p for p in input_path.rglob("*") if p.suffix.lower() in audio_exts]

print(f"Found {len(audio_files)} audio files in {input_path}")
audio_files[:5]

Found 413 audio files in ..\data\classical_resample


[WindowsPath('../data/classical_resample/00000_Preludes Book 2 -.wav'),
 WindowsPath('../data/classical_resample/00001_Preludes Book 2 - La puerta del Vino.wav'),
 WindowsPath('../data/classical_resample/00002_Sonata No 1 in F Minor Op 2 No 1 - I Allegro.wav'),
 WindowsPath('../data/classical_resample/00003_Sonata No 1 in F Minor Op 2 No 1 - II Adagio.wav'),
 WindowsPath('../data/classical_resample/00004_Sonata No 1 in F Minor Op 2 No 1 - III Menuetto Al.wav')]

## 3) Load Audio and Extract Sample Rate

In [3]:
rows = []

for i, file_path in enumerate(audio_files, 1):
    try:
        info = sf.info(file_path)  # reads metadata without loading full audio
        sample_rate = info.samplerate
        frames = info.frames
        rows.append({
            "file": file_path.name,
            "path": str(file_path),
            "sample_rate": sample_rate,
            "frames": frames,
        })
        if i % 50 == 0:
            print(f"Processed {i}/{len(audio_files)}")
    except Exception as exc:
        rows.append({
            "file": file_path.name,
            "path": str(file_path),
            "sample_rate": None,
            "frames": None,
            "error": str(exc),
        })

print(f"Collected metadata for {len(rows)} files")

Processed 50/413
Processed 100/413
Processed 150/413
Processed 200/413
Processed 250/413
Processed 300/413
Processed 350/413
Processed 400/413
Collected metadata for 413 files


## 4) Compute Duration and Build Summary Table

In [4]:
df = pd.DataFrame(rows)

# duration in seconds (frames / sample_rate)
df["duration_sec"] = df.apply(
    lambda r: (r["frames"] / r["sample_rate"]) if r.get("frames") and r.get("sample_rate") else None,
    axis=1,
 )

df = df[["file", "sample_rate", "duration_sec", "path"] + (["error"] if "error" in df.columns else [])]

df.head(413)

,file,sample_rate,duration_sec,path
0,00000_Preludes Book 2 -.wav,44100,30.002698,..\data\classical_resample\00000_Preludes Book...
1,00001_Preludes Book 2 - La puerta del Vino.wav,44100,30.002698,..\data\classical_resample\00001_Preludes Book...
2,00002_Sonata No 1 in F Minor Op 2 No 1 - I All...,44100,29.976576,..\data\classical_resample\00002_Sonata No 1 i...
3,00003_Sonata No 1 in F Minor Op 2 No 1 - II Ad...,44100,29.976576,..\data\classical_resample\00003_Sonata No 1 i...
4,00004_Sonata No 1 in F Minor Op 2 No 1 - III M...,44100,29.976576,..\data\classical_resample\00004_Sonata No 1 i...
...,...,...,...,...
408,00408_Manuel de Falla - Homenaje pour le tombe...,44100,29.976576,..\data\classical_resample\00408_Manuel de Fal...
409,00409_Camille Saint Saens - Carnaval des anima...,44100,29.976576,..\data\classical_resample\00409_Camille Saint...
410,00410_Camille Saint Saens - Carnaval des anima...,44100,29.976576,..\data\classical_resample\00410_Camille Saint...
411,00411_Camille Saint Saens - Carnaval des anima...,44100,29.976576,..\data\classical_resample\00411_Camille Saint...


## 5) Display Results and Save to CSV

In [5]:
display(df)

if output_csv is not None:
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"Saved CSV to: {output_csv}")

,file,sample_rate,duration_sec,path
0,00000_Preludes Book 2 -.wav,44100,30.002698,..\data\classical_resample\00000_Preludes Book...
1,00001_Preludes Book 2 - La puerta del Vino.wav,44100,30.002698,..\data\classical_resample\00001_Preludes Book...
2,00002_Sonata No 1 in F Minor Op 2 No 1 - I All...,44100,29.976576,..\data\classical_resample\00002_Sonata No 1 i...
3,00003_Sonata No 1 in F Minor Op 2 No 1 - II Ad...,44100,29.976576,..\data\classical_resample\00003_Sonata No 1 i...
4,00004_Sonata No 1 in F Minor Op 2 No 1 - III M...,44100,29.976576,..\data\classical_resample\00004_Sonata No 1 i...
...,...,...,...,...
408,00408_Manuel de Falla - Homenaje pour le tombe...,44100,29.976576,..\data\classical_resample\00408_Manuel de Fal...
409,00409_Camille Saint Saens - Carnaval des anima...,44100,29.976576,..\data\classical_resample\00409_Camille Saint...
410,00410_Camille Saint Saens - Carnaval des anima...,44100,29.976576,..\data\classical_resample\00410_Camille Saint...
411,00411_Camille Saint Saens - Carnaval des anima...,44100,29.976576,..\data\classical_resample\00411_Camille Saint...


Saved CSV to: ..\data\audio_metadata3.csv
